# 3.1 Code Brief: Introduction to Tree-Based ModelsThis notebook contains a condensed reference of the key code patterns from notebook 3.1. Use it as a quick reference.

## Key Pattern: Instantiate → Fit → Predict```pythonfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom xgboost import XGBClassifier# Same three lines for any model:model = ModelClass(**params)model.fit(X_train, y_train)y_prob = model.predict_proba(X_test)[:, 1]```

## Demo: Same Pattern, Three Models

In [ ]:
from sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom xgboost import XGBClassifier# All three follow the EXACT same API:models = {    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,                              use_label_encoder=False, eval_metric='logloss', random_state=42)}# Same workflow for each:for name, model in models.items():    print(f"{name}:")    print(f"  model.fit(X_train, y_train)")    print(f"  model.predict(X_test)")    print(f"  model.predict_proba(X_test)")    print()print("That's it. Same three lines. Different model, same interface.")

## How Trees Make Splits: Gini Impurity$$\text{Gini} = 1 - \sum_{i=1}^{C} p_i^2$$- **Pure node** (all one class) → Gini = 0- **Binary 50/50 split** → Gini = 0.5 (max, for 2 classes)- Tree picks the split with the largest **Gini gain** (parent impurity − weighted child impurity) — a **greedy** choice at each node**Worked example** (10 students, 6 Retained / 4 Departed, parent Gini = 1 − (0.6²+0.4²) = 0.48):| Candidate split | Weighted child Gini | ΔGini ||:---|:---|:---|| First-term GPA ≥ 2.5 | 0.16 | **0.32** (best — tree splits here) || Lives on campus (yes/no) | 0.48 | 0.00 (no information gained) |

## Controlling Overfitting| Parameter | What It Does | Typical Range ||:----------|:-------------|:-------------|| `max_depth` | Maximum tree depth | 3–15 || `min_samples_split` | Min samples to split a node | 5–50 || `min_samples_leaf` | Min samples in a leaf | 3–20 || `max_features` | Features considered per split | `'sqrt'`, `'log2'` |

## Random Forest: BaggingTrain `n_estimators` trees on bootstrap samples (random samples w/ replacement), each considering a random feature subset per split, then majority-vote the predictions. Reduces **variance**.| Parameter | What It Does | Typical Range ||:----------|:-------------|:-------------|| `n_estimators` | Number of trees | 100–500 || `max_depth` | Max depth per tree | 8–20, or None || `max_features` | Features per split | `'sqrt'` (default) || `min_samples_leaf` | Min samples in leaf | 1–10 || `class_weight` | Handle class imbalance | `'balanced'` |

## XGBoost: BoostingTrees trained **sequentially** — each new tree predicts the previous trees' errors (residuals), scaled by a learning rate $\eta$: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$. Reduces **bias**.**Walked example** (target GPA 3.4, start 2.80, $\eta$ = 0.3):| Round | Prediction | Error left | Tree says | Add $\eta \cdot h_m$ | New prediction ||:---|:---|:---|:---|:---|:---|| start | 2.80 | 0.60 | — | — | 2.80 || 1 | 2.80 | 0.60 | +1.0 | +0.30 | 3.10 || 2 | 3.10 | 0.30 | +0.6 | +0.18 | 3.28 || 3 | 3.28 | 0.12 | +0.4 | +0.12 | 3.40 || Parameter | What It Does | Typical Range ||:----------|:-------------|:-------------|| `n_estimators` | Number of boosting rounds | 100–1000 || `learning_rate` | Step size shrinkage | 0.01–0.3 || `max_depth` | Max depth per tree (keep shallow!) | 3–8 || `subsample` | Row sampling ratio | 0.7–1.0 || `colsample_bytree` | Column sampling ratio | 0.7–1.0 || `scale_pos_weight` | Handle class imbalance | ratio of neg/pos |

## Comparing the Three| Aspect | Decision Tree | Random Forest | XGBoost ||:-------|:-------------|:-------------|:--------|| Strategy | Single tree | Many trees, parallel | Trees, sequential || Reduces | — | Variance | Bias || Interpretability | Excellent | Moderate | Lower || Preprocessing needed | None | None | None || Typical use | Stakeholder communication | Reliable default | Best performance |**Key takeaway:** All three follow `instantiate → fit → predict`. No feature scaling needed for any of them.